In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
    "./test/modules/whisper_streaming"
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path
import librosa
import time

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.libri_speech_asr_corpus import *
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *
from sj_utils.string_utils import *

In [ ]:
from whisper_online import FasterWhisperASR, OnlineASRProcessor

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test-other/LibriSpeech/test-other/"
DESTINATION = "/workspaces/dev/output/LibriSpeechASRcorpus/sclient/whisper_streaming/test-other/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [ ]:
src = Path(SOURCE)
dest = Path(DESTINATION)
dest.mkdir(parents=True, exist_ok=True)

In [ ]:
asr = FasterWhisperASR("en", MODEL_SIZE)
asr.use_vad()
online = OnlineASRProcessor(asr)

In [ ]:
def transcriber(flac:Path) -> TRNFormat:
    audio, _ = librosa.load(flac, sr=SAMPLE_RATE)
    online.init()

    full_text = ""
    print(f"Processing")
    print(f"\tAudio name: {flac.name}")
    print(f"\tAudio length: {len(audio) / SAMPLE_RATE:.2f} seconds")
    start_time = time.perf_counter()
    for segment in segment_audio(audio):
        online.insert_audio_chunk(segment)
        _, _, text = online.process_iter()
        full_text += text
    _, _, text = online.finish()
    full_text += text
    end_time = time.perf_counter()
    print(f"\tProcessed time: {end_time - start_time:.2f} seconds")

    return TRNFormat(
        id = flac.stem,
        text = normalize_text_only_en(full_text).upper()
    )

In [ ]:
%%time
make_all_ref_and_hyp(src, dest, transcriber, 1)
# CPU times: user 5h 39min 24s, sys: 11min 32s, total: 5h 50min 57s
# Wall time: 1h 13min 21s

In [ ]:
concat_trn_file(
    list(sorted(p for p in dest.rglob("*.ref.trn"))),
    dest / "concat.ref.trn"
)
concat_trn_file(
    list(sorted(p for p in dest.rglob("*.hyp.trn"))),
    dest / "concat.hyp.trn"
)

In [ ]:
output = sclite_trn_run(
    dest / "concat.ref.trn",
    dest / "concat.hyp.trn",
)

In [ ]:
parse_sclite_summary(output)

# {'num_sentences': 96,
#  'num_words': 1472,
#  'correct_percent': 92.3,
#  'substitution_percent': 6.3,
#  'deletion_percent': 1.5,
#  'insertion_percent': 5.0,
#  'wer_percent': 12.8,
#  'sentence_error_percent': 60.4}